# Actividad 19. Enjambre de partículas
**Referencia:** Benítez Iglesias, R. (2014). *Inteligencia artificial avanzada*. Barcelona, Spain: Editorial UOC.

**Función a minimizar (Six-Hump Camel):**

$$f(x, y) = x^2\left(4 - 2.1x^2 + \frac{x^4}{3}\right) + xy + y^2(-4 + 4y^2)$$

**Dominio:** $x \in [-2, 2]$, $y \in [-1, 1]$

## 1. Solución con programación estructurada

Se utiliza una búsqueda por cuadrícula (grid search) evaluando la función en muchos puntos uniformemente distribuidos dentro del dominio para encontrar el mínimo.

In [45]:
import numpy as np

# Función objetivo
def funcion_np(x, y):
    sum1 = x**2 * (4 - 2.1*x**2 + x**4/3.0)
    sum2 = x*y
    sum3 = y**2 * (-4 + 4*y**2)
    return sum1 + sum2 + sum3

# Grid search vectorizado
n_puntos = 1000
xs = np.linspace(-2, 2, n_puntos)
ys = np.linspace(-1, 1, n_puntos)
X, Y = np.meshgrid(xs, ys)
Z = funcion_np(X, Y)

# Encontrar el mínimo
idx = np.unravel_index(np.argmin(Z), Z.shape)
mejor_x = X[idx]
mejor_y = Y[idx]
mejor_valor = Z[idx]

print('=== Resultado con programación estructurada (Grid Search) ===')
print(f'x = {mejor_x:.6f}')
print(f'y = {mejor_y:.6f}')
print(f'f(x,y) = {mejor_valor:.6f}')
print(f'Total de puntos evaluados: {n_puntos}x{n_puntos} = {n_puntos**2:,}')

=== Resultado con programación estructurada (Grid Search) ===
x = 0.090090
y = -0.711712
f(x,y) = -1.031621
Total de puntos evaluados: 1000x1000 = 1,000,000


## 2. Solución con algoritmo de enjambre de partículas (PSO)

Implementación basada en el código del libro (página 248).

In [46]:
from random import random

# Función objetivo
def funcion(x, y):
    sum1 = x**2 * (4 - 2.1*x**2 + x**4/3.0)
    sum2 = x*y
    sum3 = y**2 * (-4 + 4*y**2)
    return sum1 + sum2 + sum3

# Devuelve un número aleatorio dentro de un rango con
# distribución uniforme (proporcionada por random)
def aleatorio(inf, sup):
    return random()*(sup-inf) + inf

# Clase que representa una partícula individual y que facilita
# las operaciones necesarias
class Particula:
    # Algunos atributos de clase (comunes a todas las partículas)
    # Parámetros para actualizar la velocidad
    inercia    = 1.4
    cognitiva  = 2.0
    social     = 2.0
    # Límites del espacio de soluciones
    infx = -2.0
    supx =  2.0
    infy = -1.0
    supy =  1.0
    # Factor de ajuste de la velocidad inicial
    ajusteV = 100.0

    # Crea una partícula dentro de los límites indicados
    def __init__(self):
        self.x  = aleatorio(Particula.infx, Particula.supx)
        self.y  = aleatorio(Particula.infy, Particula.supy)
        self.vx = aleatorio(Particula.infx/Particula.ajusteV,
                            Particula.supx/Particula.ajusteV)
        self.vy = aleatorio(Particula.infy/Particula.ajusteV,
                            Particula.supy/Particula.ajusteV)
        self.xLoc     = self.x
        self.yLoc     = self.y
        self.valorLoc = funcion(self.x, self.y)

    # Actualiza la velocidad de la partícula
    def actualizaVelocidad(self, xGlob, yGlob):
        cogX    = Particula.cognitiva*random()*(self.xLoc-self.x)
        socX    = Particula.social*random()*(xGlob-self.x)
        self.vx = Particula.inercia*self.vx + cogX + socX
        cogY    = Particula.cognitiva*random()*(self.yLoc-self.y)
        socY    = Particula.social*random()*(yGlob-self.x)
        self.vy = Particula.inercia*self.vy + cogY + socY

    # Actualiza la posición de la partícula
    def actualizaPosicion(self):
        self.x = self.x + self.vx
        self.y = self.y + self.vy

        # Debe mantenerse dentro del espacio de soluciones
        self.x = max(self.x, Particula.infx)
        self.x = min(self.x, Particula.supx)
        self.y = max(self.y, Particula.infy)
        self.y = min(self.y, Particula.supy)

        # Si es inferior a la mejor, la adopta como mejor
        valor = funcion(self.x, self.y)
        if valor < self.valorLoc:
            self.xLoc     = self.x
            self.yLoc     = self.y
            self.valorLoc = valor


# Mueve un enjambre de partículas durante las iteraciones indicadas.
# Devuelve las coordenadas y el valor del mínimo obtenido.
def enjambreParticulas(particulas, iteraciones, reduccionInercia):

    # Registra la mejor posición global y su valor
    mejorParticula = min(particulas, key=lambda p:p.valorLoc)
    xGlob     = mejorParticula.xLoc
    yGlob     = mejorParticula.yLoc
    valorGlob = mejorParticula.valorLoc

    # Bucle principal de simulación
    for iter in range(iteraciones):
        # Actualiza la velocidad y posición de cada partícula
        for p in particulas:
            p.actualizaVelocidad(xGlob, yGlob)
            p.actualizaPosicion()

        # Hasta que no se han movido todas las partículas no se
        # actualiza el mínimo global, para simular que todas se
        # mueven a la vez
        mejorParticula = min(particulas, key=lambda p:p.valorLoc)
        if mejorParticula.valorLoc < valorGlob:
            xGlob     = mejorParticula.xLoc
            yGlob     = mejorParticula.yLoc
            valorGlob = mejorParticula.valorLoc

        # Finalmente se reduce la inercia de las partículas
        Particula.inercia *= reduccionInercia

    return (xGlob, yGlob, valorGlob)


# Parámetros del problema
nParticulas = 10
iteraciones = 100
redInercia  = 0.9

# Genera un conjunto inicial de partículas
particulas = [Particula() for i in range(nParticulas)]

# Ejecuta el algoritmo del enjambre de partículas
resultado = enjambreParticulas(particulas, iteraciones, redInercia)

print('=== Resultado con PSO ===')
print(f'x = {resultado[0]:.6f}')
print(f'y = {resultado[1]:.6f}')
print(f'f(x,y) = {resultado[2]:.6f}')
print(f'Total de evaluaciones: {nParticulas} partículas x {iteraciones} iteraciones = {nParticulas*iteraciones:,}')

=== Resultado con PSO ===
x = -0.086410
y = 0.704382
f(x,y) = -1.031057
Total de evaluaciones: 10 partículas x 100 iteraciones = 1,000


## 3. Comparación de resultados

El mínimo global teórico de la función Six-Hump Camel es **f(x,y) = -1.0316** y se alcanza en dos puntos simétricos:
- $(0.0898, -0.7126)$
- $(-0.0898, 0.7126)$

In [47]:
print('=== Comparación de resultados ===')
print(f'{"Método":<30} {"x":>10} {"y":>10} {"f(x,y)":>12}')
print('-' * 65)
print(f'{"Grid Search":<30} {mejor_x:>10.6f} {mejor_y:>10.6f} {mejor_valor:>12.6f}')
print(f'{"PSO (Enjambre)":<30} {resultado[0]:>10.6f} {resultado[1]:>10.6f} {resultado[2]:>12.6f}')
print(f'{"Mínimo teórico":<30} {0.0898:>10.4f} {-0.7126:>10.4f} {-1.0316:>12.4f}')
print()
print('Observaciones:')
print('- El grid search encuentra una buena aproximación evaluando el dominio completo.')
print('- El PSO converge al mínimo con muchas menos evaluaciones de la función.')
print('- Ambos métodos se acercan al valor teórico de -1.0316.')
print(f'- Grid search: {n_puntos**2:,} evaluaciones vs PSO: {nParticulas*iteraciones:,} evaluaciones.')

=== Comparación de resultados ===
Método                                  x          y       f(x,y)
-----------------------------------------------------------------
Grid Search                      0.090090  -0.711712    -1.031621
PSO (Enjambre)                  -0.086410   0.704382    -1.031057
Mínimo teórico                     0.0898    -0.7126      -1.0316

Observaciones:
- El grid search encuentra una buena aproximación evaluando el dominio completo.
- El PSO converge al mínimo con muchas menos evaluaciones de la función.
- Ambos métodos se acercan al valor teórico de -1.0316.
- Grid search: 1,000,000 evaluaciones vs PSO: 1,000 evaluaciones.
